# ASTR 457: The Midterm

Posted Thu Oct 8, due Wed Oct 14 by Noon (fork → PR, as always). Graded like a lab.

Three parts. Each is designed to test something every lab so far has come back to - which error bar is the right one, whether the model needs the extra parameter, and whether the check you ran could have failed. You've now got the tools to do all of this, so let's do it on numbers you haven't seen: 250 real Type Ia supernovae, and someone else's analysis of real white dwarfs.

Part 1 is 30 pts (6 each), Part 2 is 45 pts, and Part 3 is 25 pts. You'll need 2b and 2c right to get 2d right.

Remember, the rules are the syllabus rules - open book, you can use AI tools (document them in 3b), you can work together and discuss problems and solutions, but you have to write up your own solution and run it on **your own data** in `data/<your netid>/` (nobody else's numbers will match yours).

Put your notebook in `submissions/<your netid>/midterm.ipynb`. The path below (`DATA`) reaches your files from there. Make sure it runs top to bottom from a fresh kernel ("Restart & Run All") - that is how it will be graded, and how it will be run if you defend it.

Throughout, make sure every number you quote has a unit, an error bar and a sensible number of digits (two significant figures on the error bar, and round the value to match).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

NETID = 'nraja3'   # <-- set this
DATA = f'../../labs/midterm/data/{NETID}/'   # from submissions/<your netid>/

def load(name):
    """Read one of your CSV files as a pandas table; d['column'].to_numpy() gives a numpy array. Names stay strings."""
    return pd.read_csv(DATA + name + '.csv', dtype={'name': str})

## Part 1: Error bars (30 pts, 6 each)

Five short scenarios. Each one is a number someone handed you - say whether it's right, compute the right one, and say in a sentence or two why. Show the code. Each of these should be about 20 minutes of work.

### 1a - The extinction coefficient (`p1a_extinction.csv`)

An observer measured a photometric zero point at 15 different airmasses through one night. Extinction in magnitudes grows in proportion to the airmass $X$, so the zero point should follow a line, $m = k\,X + m_0$, where $k$ is the extinction coefficient in mag per airmass. Columns: `airmass`, `zeropoint_mag`, `sigma_quoted_mag` (the pipeline's error on each point).

1. Fit the line by weighted least squares with the quoted errors. Report $k$ and its error bar computed from the quoted errors alone.
2. Now run `np.polyfit(airmass, zeropoint_mag, 1, w=1/sigma_quoted_mag, cov=True)` and report its error bar on $k$. Explain exactly what `polyfit` did to get a different number (read its documentation for `cov`).
3. Compute $\chi^2$ and the number of degrees of freedom. How probable is a $\chi^2$ this large or this small if the quoted errors were right? (Compute it from the $\chi^2$ distribution.)
4. Report the error bar on $k$ you'd quote, and justify it. What does your answer to (3) tell you about the pipeline's error bars?

In [ ]:
# 1a: your work here

d = load('p1a_extinction')
x = d['airmass'].to_numpy()
y = d['zeropoint_mag'].to_numpy()
s = d['sigma_quoted_mag'].to_numpy()

p, cov_given = np.polyfit(x, y, 1, w=1/s, cov='unscaled')

p2, cov_scaled = np.polyfit(x, y, 1, w=1/s, cov=True)
k = p[0]
err_given = np.sqrt(cov_given[0, 0])
err_scaled = np.sqrt(cov_scaled[0, 0])
chi2 = np.sum(((y - np.polyval(p, x))/s)**2)
dof = len(y) - 2
p_high = stats.chi2.sf(chi2, dof)
p_low = stats.chi2.cdf(chi2, dof)
p_two = min(1, 2*min(p_high, p_low))
print('Fitted line slope:', k, '+/-', err_given, 'mag per airmass')
print('polyfit cov=True slope error:', err_scaled, 'mag per airmass')
print('chi-square:', chi2, 'dof:', dof, 'reduced chi-square:', chi2/dof)
print('P(chi2 >= observed):', p_high, 'P(chi2 <= observed):', p_low)
print('Two-tailed unusualness:', p_two)
print('Suggested error based on residual scatter:', err_scaled, 'mag per airmass')
plt.errorbar(x, y, yerr=s, fmt='o')
plt.plot(np.sort(x), np.polyval(p, np.sort(x)))
plt.xlabel('Airmass'); plt.ylabel('Zero point (mag)'); plt.show()

**ANSWER:**

*I got a larger error bar of 0.024 mag per airmass because polyfit multiplies the covariance matrix by the reduced chi-square to account for the scatter in the data. The reduced chi-square was 4.83, so the error bar increased by a factor of about 2.20.
I calculated chi-square = 62.82 with 13 degrees of freedom. The probability of getting a chi-square this large or larger is only 1.64x10^-8, which is extremely small. This means the observed scatter is much larger than expected if the quoted errors and linear model are correct.*

### 1b - The centroid (`p1b_centroid.csv`)

A classmate measured the position of a star from `n_photons` photons, each landing at a position drawn from a Gaussian of width `psf_sigma_arcsec` around the true position. They report an error on the position of `quoted_sigma_x0_arcsec`, from a clever weighting scheme they found online.

1. Compute the Cramér-Rao bound on the error of any unbiased estimator of the position (Lab 02).
2. Is the quoted error possible? If not, say what the classmate's number must mean instead, and name two concrete mistakes that would produce an error bar this small.

In [ ]:
# 1b: your work here

d = load('p1b_centroid')
print(d.to_string(index=False))
for _, row in d.iterrows():
    nphot = float(row['n_photons'])
    width = float(row['psf_sigma_arcsec'])
    quoted = float(row['quoted_sigma_x0_arcsec'])
    bound = width/np.sqrt(nphot)
    print('CR bound =', bound, 'arcsec; quoted =', quoted, 'arcsec; quoted/bound =', quoted/bound)


**ANSWER:**

*The Cramér-Rao bound is 0.0460 arcsec, but the classmate reported 0.0297 arcsec. Their error is too small to be the standard deviation of an unbiased estimator under this model. They may have counted photons more than once or used the wrong PSF width*

### 1c - The transit depth's error bar (`p1c_profile.csv`)

A transit-depth fit produced this profile: columns `depth_ppt` (the depth, in parts per thousand) and `minus2lnL` ($-2\ln L$ at that depth, other parameters fixed at their best values). A classmate took the second derivative of `minus2lnL` at its minimum and reported $\sigma = 1/\sqrt{d^2(-2\ln L)/d\,\textrm{depth}^2}$.

1. Find the best depth and the correct $1\sigma$ error from the curvature. Is the classmate's error bar right? If not, by what factor is it off, and why?
2. Read the $\Delta(-2\ln L) = 1$ interval directly off the profile. Is it symmetric? Compare it with your curvature answer.
3. If the classmate quotes $\pm$ their $\sigma$ on many independent datasets, what fraction of those intervals contain the true depth? Predict it, then check it with a simulation (a few thousand fake best-fit depths drawn around a truth you choose, using your correct $\sigma$).

In [ ]:
# 1c: your work here

d = load('p1c_profile').sort_values('depth_ppt')
x = d['depth_ppt'].to_numpy()
y = d['minus2lnL'].to_numpy()
i = np.argmin(y)

ix = np.argsort(abs(x-x[i]))[:7]
quad = np.polyfit(x[ix]-x[i], y[ix]-y[i], 2)
best = x[i] - quad[1]/(2*quad[0])
second_derivative = 2*quad[0]
sigma = np.sqrt(2/second_derivative)
wrong_sigma = np.sqrt(1/second_derivative)

target = np.polyval(quad, best-x[i]) + y[i] + 1
left = np.interp(target, y[:i+1][::-1], x[:i+1][::-1])
right = np.interp(target, y[i:], x[i:])
print('Best depth:', best, 'ppt')
print('Second derivative:', second_derivative, 'ppt^-2')
print('Correct curvature sigma:', sigma, 'ppt')
print('Common missing-factor-sqrt(2) answer:', wrong_sigma, 'ppt')
print('Profile 68% interval:', left, 'to', right, 'ppt')
print('Lower/upper errors:', best-left, right-best, 'ppt')
print('Expected coverage using wrong sigma:', 2*stats.norm.cdf(wrong_sigma/sigma)-1)
rng = np.random.default_rng(457)
truth = best
fake_best = rng.normal(truth, sigma, size=5000)
coverage = np.mean(np.abs(fake_best-truth) <= wrong_sigma)
print('Simulated wrong-error coverage:', coverage)
plt.plot(x,y); plt.axhline(target,ls='--'); plt.axvline(best,ls=':')
plt.xlabel('Depth (ppt)');plt.ylabel('-2 ln L');plt.show()

**ANSWER:**

*The best depth is 2.961 +/- 0.277 ppt. The classmate's error of 0.196 ppt is too small by a factor of sqrt{2} because they missed a factor of 2 in the curvature formula. The 68% interval is 2.682–3.236 ppt, which is almost symmetric. The predicted coverage using the wrong error is 52.0%, and my simulation gave 52.2%.*

### 1d - The companion mass (`p1d_mass_samples.csv`)

These are 8000 posterior samples of the mass of an unseen companion, `M2_Msun`. Three classmates report its uncertainty as (i) the variance of the samples, (ii) the 84th percentile minus the 16th, and (iii) the standard deviation divided by $\sqrt{8000}$.

1. Report the mass as a median with its 16th-84th percentile error bars, in solar masses.
2. For each of (i)-(iii): what quantity is it actually, what are its units, and by roughly what factor is it wrong as a $1\sigma$ error bar?
3. Is a symmetric $\pm$ error bar appropriate for this posterior? If not, show why.

In [ ]:
# 1d: your work here

d=load('p1d_mass_samples')
m=d['M2_Msun'].to_numpy()
q16, med, q84=np.percentile(m,[16,50,84])
minus=med-q16; plus=q84-med
sd=np.std(m,ddof=1)
var=np.var(m,ddof=1)
range68=q84-q16
sem=sd/np.sqrt(len(m))
print('Mass =',med,'+',plus,'-',minus,'solar masses')
print('Standard deviation:',sd,'solar masses')
print('(i) Variance:',var,'solar masses squared')
print('(ii) 84th-16th:',range68,'solar masses; ratio to sd:',range68/sd)
print('(iii) sd/sqrt(N):',sem,'solar masses; factor too small:',sd/sem)
plt.hist(m,bins=40);plt.axvline(q16,ls='--');plt.axvline(med);plt.axvline(q84,ls='--')
plt.xlabel('Companion mass (solar masses)');plt.ylabel('Samples');plt.show()

**ANSWER:**

The companion mass is:

$$
M_2 = 7.25^{+1.61}_{-1.36}\ M_\odot
$$

(i) The variance is $2.33\ M_\odot^2$, which has the wrong units for an error bar.

(ii) The percentile difference is $2.97\ M_\odot$, which is the full 68% interval width, approximately twice a one-sided error.

(iii) The standard error is $0.0171\ M_\odot$, about 89 times smaller than the posterior standard deviation.

A symmetric error bar is not appropriate because the distribution has a longer tail toward higher masses.

### 1e - The slope at fixed intercept (`p1e_slope_intercept_samples.csv`)

Posterior samples of a slope and an intercept from a line fit. A classmate fixed the intercept at its best value, refit only the slope, and quoted the slope's error from that 1-D fit.

1. Compute the slope's error bar the classmate got (the spread of the slope *at fixed intercept*), and the marginal one (the spread of the slope with the intercept marginalized over).
2. Compute their ratio, and the correlation coefficient $\rho$ of the samples. How does the ratio depend on $\rho$? Plot the samples and draw both error bars on the plot.
3. When is the classmate's number the right one to quote? One sentence.

In [ ]:
# 1e: your work here

d=load('p1e_slope_intercept_samples')
print('Available columns:',list(d.columns))

slope_col=next(c for c in d.columns if 'slope' in c.lower() or c.lower()=='alpha')
intercept_col=next(c for c in d.columns if 'intercept' in c.lower() or c.lower()=='beta')
a=d[slope_col].to_numpy(); b=d[intercept_col].to_numpy()
rho=np.corrcoef(a,b)[0,1]
sd_a=np.std(a,ddof=1)
fixed=sd_a*np.sqrt(1-rho**2)
print('Marginal slope error:',sd_a)
print('Fixed-intercept slope error:',fixed)
print('fixed / marginal:',fixed/sd_a)
print('Correlation:',rho)
plt.scatter(b,a,s=2,alpha=.2)
center_b=np.median(b);center_a=np.median(a)
plt.errorbar(center_b,center_a,yerr=fixed,fmt='o',label='fixed intercept')
plt.errorbar(center_b+0.1*np.std(b),center_a,yerr=sd_a,fmt='o',label='marginal')
plt.xlabel('Intercept');plt.ylabel('Slope');plt.legend();plt.show()

**ANSWER:**

*The fixed-intercept uncertainty is 0.0223, while the marginal uncertainty is 0.0401. Their ratio is 0.555, and the correlation is $\rho=-0.832$. The ratio follows $\sqrt{1-\rho^2}$, so stronger correlations make the fixed-intercept error smaller. The classmate's error is only appropriate if the intercept is known exactly.*

## Part 2: 250 supernovae and a step (45 pts)

Type Ia supernovae are standardizable candles: their peak brightness, corrected for how fast they fade and for their color, measures distance well enough to have discovered cosmic acceleration. Your file `p2_pantheon_sne.csv` holds 250 real SNe Ia from the Pantheon+ compilation (Scolnic et al. 2022, ApJ 938, 113; [Brout et al. 2022](https://arxiv.org/abs/2202.04077), ApJ 938, 110), all at redshift $0.023 < z < 0.15$. For each supernova, a light-curve fit (SALT2) gives:

- `mB`: the peak apparent magnitude in the B band, and `mB_err`;
- `x1`: the light-curve *stretch* (broader light curves are brighter), and `x1_err`;
- `c`: the *color* (redder supernovae are fainter, from dust and intrinsically), and `c_err`;
- `cov_mB_x1`, `cov_mB_c`, `cov_x1_c`: the covariances between those three, from the same light-curve fit;
- `z`: the redshift; `host_logmass`, `host_logmass_err`: $\log_{10}$ of the host galaxy's stellar mass in solar masses, and its error.

The standardized brightness is the **Tripp relation**,

$$m_B = M + \mu(z) - \alpha\,x_1 + \beta\,c + \gamma\,\Theta(\log M_\star - 10),$$

where $M$ is the absolute magnitude of a "standard" SN Ia, $\mu(z)$ the distance modulus, $\Theta$ is 1 for hosts more massive than $10^{10}\,M_\odot$ and 0 otherwise, and $\gamma$ is the **host-mass step**. Supernovae in massive hosts appear to be brighter after standardization (Kelly et al. 2010; Sullivan et al. 2010); whether the step is real, and why, matters for supernova dark-energy measurements. In this form, a step that makes supernovae in massive hosts brighter has $\gamma < 0$; the published value is about $\gamma \approx -0.06$ mag. So the question for your sample is whether you need $\gamma$ at all.

There are two more things you'll need. At these redshifts the galaxies' own motions (peculiar velocities, about 250 km/s) shift $z$, and so $\mu$; add $\sigma_{\mu,\rm pec} = \frac{5}{\ln 10}\frac{250\ {\rm km/s}}{c\,z}$ to each supernova's variance. And no supernova is a perfect standard candle: add an intrinsic scatter $\sigma_{\rm int}$, a free parameter. Use a flat $\Lambda$CDM distance modulus with $H_0 = 70$ km/s/Mpc and $\Omega_m = 0.3$ (`astropy.cosmology.FlatLambdaCDM(H0=70, Om0=0.3).distmod(z)`); $M$ absorbs the choice of $H_0$.

We planted between **two and four contaminants** in your file - objects whose `mB` we made fainter by more than a magnitude, so they are not standard SNe Ia any more (think of a mis-classified peculiar supernova). Their names are not in the file. Real data have a few real outliers too, so not everything far from the line is planted.

### 2a - Look at the data (5 pts)

1. Compute $\mu(z)$ and plot the **Hubble residuals** $m_B - \mu(z)$ against $x_1$, against $c$, and against `host_logmass`.
2. Which way do the $x_1$ and $c$ trends go? Is that the sign the Tripp relation expects? Two or three sentences.
3. The raw trend with host mass may not have the sign of $\gamma$. Why not? (Plot $x_1$ against host mass.)
4. Can you see any of the contaminants?
5. What units do $\alpha$ and $\beta$ have?

In [ ]:
# 2a: your work here

from astropy.cosmology import FlatLambdaCDM
sn=load('p2_pantheon_sne')
print('Columns:',list(sn.columns))
cosmo=FlatLambdaCDM(H0=70,Om0=0.3)
z=sn['z'].to_numpy(); mu=cosmo.distmod(z).value
mb=sn['mB'].to_numpy(); x1=sn['x1'].to_numpy(); color=sn['c'].to_numpy()
host=sn['host_logmass'].to_numpy()
hr=mb-mu
fig,axs=plt.subplots(1,4,figsize=(15,3))
for ax, xx, xlabel in zip(axs,[x1,color,host,x1],['x1','Color c','Host log mass','x1']):
    yy=host if xlabel=='x1' and ax is axs[3] else hr
    ax.scatter(xx,yy,s=12)
    ax.set_xlabel(xlabel);ax.set_ylabel('Host mass' if ax is axs[3] else 'mB - mu (mag)')
plt.tight_layout();plt.show()
for label,xx in [('x1',x1),('c',color),('host mass',host)]:
    print('Slope of raw residual vs',label,np.polyfit(xx,hr,1)[0])
print('Host vs x1 slope:',np.polyfit(host,x1,1)[0])
print('Top bright/faint residual rows:')
print(sn.iloc[np.argsort(hr)[-8:]][['mB','x1','c','host_logmass']])

**ANSWER:**

*The residuals decrease with $x_1$ (slope -0.159) and increase with color (slope +2.90). Both agree with the Tripp relation. The raw host-mass trend is positive because host mass and stretch are negatively correlated. I identified three possible contaminants at rows 40, 110, and 132. The residuals are measured in magnitudes, while $x_1$ and $c$ are dimensionless, so $\alpha$ and $\beta$ have magnitude units.*

### 2b - The likelihood (10 pts)

The parameters are $\theta = (M, \alpha, \beta, \gamma, \sigma_{\rm int})$. Each supernova's variance is

$$\sigma_i^2 = \mathbf{v}^T C_i\,\mathbf{v} + \sigma_{\mu,{\rm pec},i}^2 + \sigma_{\rm int}^2, \qquad \mathbf{v} = (1, \alpha, -\beta),$$

where $C_i$ is that supernova's $3\times3$ covariance of $(m_B, x_1, c)$.

1. Write out $\mathbf{v}^T C_i\,\mathbf{v}$ term by term, and explain in one sentence why the off-diagonal terms are there.
2. Write the log-likelihood. Remember the $\ln\sigma_i^2$ term - the variance depends on the parameters.
3. State a prior for every parameter, and justify each in one line.
4. Decide whether to sample $\ln\sigma_{\rm int}$ or $\sigma_{\rm int}$, and say why.
5. Decide what to do about the contaminants. You may model them or remove them, but say how you found them and what a cut would cost you.

In [ ]:
# 2b: your work here

smb=sn['mB_err'].to_numpy();sx=sn['x1_err'].to_numpy();sc=sn['c_err'].to_numpy()
cmx=sn['cov_mB_x1'].to_numpy();cmc=sn['cov_mB_c'].to_numpy();cxc=sn['cov_x1_c'].to_numpy()
I=(host>10).astype(float)

bounds=[(-21,-17),(0,.5),(0,6),(-.5,.5),(.01,.7)]

def neglog(p, mask=None, step=True):
    if mask is None: mask=np.ones(len(mb),dtype=bool)
    M,alpha,beta,delta,sig=p
    if not step: delta=0
    if any(v<=lo or v>=hi for v,(lo,hi) in zip(p,bounds)):return np.inf
    v=smb**2+alpha**2*sx**2+beta**2*sc**2+2*alpha*cmx-2*beta*cmc-2*alpha*beta*cxc
    v=v+sig**2+(5/np.log(10)*250/(299792.458*z))**2
    if np.any(v[mask]<=0):return np.inf
    pred=mu+M-alpha*x1+beta*color+delta*I
    r=mb-pred
    return .5*np.sum(r[mask]**2/v[mask]+np.log(2*np.pi*v[mask]))

start=np.array([-19.3,.14,3.0,-.06,.15])
pre=minimize(neglog,start,method='Nelder-Mead',options={'maxiter':10000})
p0=pre.x
print('Preliminary parameters:',p0)
v0=smb**2+p0[1]**2*sx**2+p0[2]**2*sc**2+2*p0[1]*cmx-2*p0[2]*cmc-2*p0[1]*p0[2]*cxc+p0[4]**2+(5/np.log(10)*250/(299792.458*z))**2
res0=(mb-(mu+p0[0]-p0[1]*x1+p0[2]*color+p0[3]*I))/np.sqrt(v0)

keep=res0<5
print('Removed rows:',np.where(~keep)[0].tolist(), 'count:',np.sum(~keep))
plt.scatter(z,res0,s=12);plt.axhline(5,ls='--');plt.xlabel('Redshift');plt.ylabel('Pre-fit standardized residual');plt.show()

**ANSWER:**

1.
$$
\begin{aligned}
\mathbf{v}^T C_i \mathbf{v} ={}&
\sigma_{m_B,i}^2
+\alpha^2\sigma_{x_1,i}^2
+\beta^2\sigma_{c,i}^2\\
&+2\alpha\operatorname{Cov}(m_B,x_1)_i\\
&-2\beta\operatorname{Cov}(m_B,c)_i\\
&-2\alpha\beta\operatorname{Cov}(x_1,c)_i
\end{aligned}
$$

The off-diagonal terms are included because the measurements come from the same light-curve fit, so their errors can be correlated.

2.

$$
\ln L=-\frac{1}{2}\sum_i
\left[
\frac{(m_{B,i}-m_{B,i}^{\mathrm{model}})^2}{\sigma_i^2}
+\ln(2\pi\sigma_i^2)
\right]
$$



3.

- $M\sim U(-21,-17)$ mag, since this covers a reasonable range of supernova absolute magnitudes.
- $\alpha\sim U(0,0.5)$, since larger stretch should make supernovae brighter.
- $\beta\sim U(0,6)$, since redder supernovae should be fainter.
- $\gamma\sim U(-0.5,0.5)$ mag, to allow either a positive or negative host-mass step.
- $\sigma_{\mathrm{int}}\sim U(0.01,0.7)$ mag, to allow positive intrinsic scatter.

4. I chose to sample $\sigma_{\mathrm{int}}$ instead of $\ln\sigma_{\mathrm{int}}$ because it is easier to interpret directly in magnitudes and matches my uniform prior.

5. I fitted all 250 supernovae first and calculated their standardized residuals. I removed three possible contaminants at rows 40, 110, and 132 because their positive residuals were greater than $5\sigma$. This left 247 supernovae. The disadvantage is that some unusual but real supernovae might also be removed, which could affect the results.

### 2c - Sampling and convergence (12 pts)

1. Find the maximum a posteriori point. Then run **four independent chains** from starting points spread wider than the posterior (you choose how, and say how you chose). Any sampler is fine - your own Metropolis, `emcee` (one ensemble per chain), or `PyMC`.
2. Report $\hat R$ and the effective sample size for every parameter (`arviz.rhat`, `arviz.ess`, or `arviz.summary`), and say what values would have made you run longer.
3. Make a corner plot, and quote $\alpha$, $\beta$, $\gamma$ and $\sigma_{\rm int}$ as medians with 16th-84th percentile error bars.
4. Which parameters are correlated strongly? Comment on why. Then fix $M$ at its MAP value, refit $\gamma$, and compare that $\sigma_\gamma$ with the marginal one (cf. 1e).

`ArviZ` 1.x (install it with `conda install -c conda-forge pymc arviz`, as in `INSTALL.txt`) takes your chains as a dictionary of arrays shaped (chain, draw): `idata = az.from_dict({'posterior': {'alpha': a, 'beta': b}})`, then `az.summary(idata)`.

In [ ]:
# 2c: your work here

def hessian(f, p, h):
    n = len(p)
    H = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            di = np.zeros(n)
            dj = np.zeros(n)

            di[i] = h[i]
            dj[j] = h[j]

            H[i, j] = (
                f(p + di + dj)
                - f(p + di - dj)
                - f(p - di + dj)
                + f(p - di - dj)
            ) / (4 * h[i] * h[j])

    return H

fit=minimize(lambda p:neglog(p,keep),p0,method='Nelder-Mead',options={'maxiter':15000})
map_p=fit.x
print('MAP (M,alpha,beta,delta,sigma_int):',map_p,'success:',fit.success)

proposal=np.array([.018,.012,.09,.018,.012])
starts=np.array([map_p+np.array([a,b,c,d,e]) for a,b,c,d,e in [
    [-.15,-.06,-.5,-.08,-.04],[.15,.06,.5,.08,.04],
    [-.1,.07,-.4,.1,.05],[.1,-.07,.4,-.1,-.05]]])

for j,(lo,hi) in enumerate(bounds):starts[:,j]=np.clip(starts[:,j],lo+.005,hi-.005)
rng=np.random.default_rng(457)
nstep=18000
chains=np.zeros((4,nstep,5))
for c in range(4):
    now=starts[c].copy();current=neglog(now,keep);accepted=0
    for t in range(nstep):
        trial=now+rng.normal(0,proposal)
        score=neglog(trial,keep)
        if np.log(rng.random())<current-score:
            now=trial;current=score;accepted+=1
        chains[c,t]=now
    print('Chain',c+1,'acceptance:',accepted/nstep)
samples=chains[:,6000::5,:]
names=['M','alpha','beta','delta','sigma_int']
print('Samples shape:',samples.shape)
import arviz as az
import xarray as xr

data = xr.Dataset({
    name: (('chain', 'draw'), samples[:, :, i])
    for i, name in enumerate(names)
})

summary = az.summary(data, var_names=names)

print(summary[['mean', 'sd', 'r_hat', 'ess_bulk']])
flat=samples.reshape(-1,5)
for name in ['alpha','beta','delta','sigma_int']:
    j=names.index(name);q=np.percentile(flat[:,j],[16,50,84]);print(name,'=',q[1],'+',q[2]-q[1],'-',q[1]-q[0])

from pandas.plotting import scatter_matrix
subset=pd.DataFrame(flat[::10],columns=names)
scatter_matrix(subset,figsize=(11,11),diagonal='hist',alpha=.12,hist_kwds={'bins':25})
plt.suptitle('Posterior corner-style plot');plt.show()
print('Correlation matrix:')
print(pd.DataFrame(np.corrcoef(flat.T),index=names,columns=names).round(2))

In [ ]:
def neg_fixed(q):
    p = np.array([map_p[0], q[0], q[1], q[2], q[3]])
    return neglog(p, keep)

fixed_fit = minimize(neg_fixed, map_p[1:], method='Nelder-Mead')

Hfixed = hessian(neg_fixed, fixed_fit.x, np.array([.002, .01, .002, .002]))

fixed_gamma_err = np.sqrt(np.linalg.inv(Hfixed)[2, 2])
marg_gamma_err = np.std(flat[:, 3], ddof=1)

print('Gamma with M fixed:', fixed_fit.x[2])
print('Fixed gamma error:', fixed_gamma_err)
print('Marginal gamma error:', marg_gamma_err)

**ANSWER:**

1. I started four independent chains from different positions spread wider than the posterior uncertainties to check if they would all reach the same results.

2. All R-hat values are around 1.00, and the effective sample sizes are above 1,000, suggesting convergence. I would run longer if R-hat was above 1.01 or the effective sample sizes were too small.

4. The strongest correlation is between $M$ and $\gamma$, with a correlation coefficient of $-0.73$. This happens because changing $M$ can partly be balanced by changing $\gamma$.

### 2d - Model comparison (10 pts)

Compare $M_0$ ($\gamma = 0$) with $M_1$ ($\gamma$ free).

1. Compute $2\ln B_{1,0}$ with the Laplace approximation (Day 13), with a prior on $\gamma$ you state and defend. How does it change if that prior is twice as wide?
2. Compute $\textrm{BIC}_0 - \textrm{BIC}_1$ for comparison.
3. How strong is your answer on Day 8's scale (0-2 weak, 2-6 positive, 6-10 strong, above 10 very strong)? Compare $\gamma$ with the published $-0.06$ mag. Could your 250 supernovae have detected a step that size? (One way: your $\sigma_\gamma$ tells you how many $\sigma$ a $-0.06$ step would be.)

In [ ]:
# 2d: your work here

b0=[bounds[j] for j in [0,1,2,4]]
def n0(q):
    return neglog(np.array([q[0],q[1],q[2],0,q[3]]),keep)
f0=minimize(n0,map_p[[0,1,2,4]],method='Nelder-Mead',options={'maxiter':15000})
f1=fit
h0=hessian(n0,f0.x,np.array([.002,.002,.01,.002]))
h1=hessian(lambda p:neglog(p,keep),f1.x,np.array([.002,.002,.01,.002,.002]))
sign0,logdet0=np.linalg.slogdet(h0);sign1,logdet1=np.linalg.slogdet(h1)
print('Hessian determinant signs:',sign0,sign1)
vol0=np.prod([hi-lo for lo,hi in b0]);vol1=np.prod([hi-lo for lo,hi in bounds])
logz0=-f0.fun-np.log(vol0)+2*np.log(2*np.pi)-.5*logdet0
logz1=-f1.fun-np.log(vol1)+2.5*np.log(2*np.pi)-.5*logdet1
logB10=logz1-logz0
bic0=2*f0.fun+4*np.log(np.sum(keep))
bic1=2*f1.fun+5*np.log(np.sum(keep))
print('log B10 (step/free divided by no-step):',logB10)
print('2 log B10:',2*logB10)
print('If delta prior twice as wide (interior posterior): log B10 about',logB10-np.log(2))
print('BIC H0:',bic0,'BIC H1:',bic1,'BIC1-BIC0:',bic1-bic0)
qdelta=np.percentile(flat[:,3],[16,50,84]);sigdelta=(qdelta[2]-qdelta[0])/2
print('Delta posterior median +/- approx sigma:',qdelta[1],sigdelta,'mag')
print('A published -0.06 mag step is',abs(-.06)/sigdelta,'sigma from zero with this sigma')

**ANSWER:**

1. I used a uniform prior of $-0.5$ to $0.5$ mag for $\gamma$ because it allows both positive and negative host-mass steps.

Using the Laplace approximation:

$$
2\ln B_{1,0}=-2.97
$$

When the prior is twice as wide, the result becomes approximately $-4.36$. This makes the step model less favored.

2. The BIC difference is:

$$
\mathrm{BIC}_0-\mathrm{BIC}_1=-2.48
$$

This means the model without a step is slightly preferred because it has the lower BIC.

3. The value of $-2.97$ indicates positive but not strong evidence for the no-step model on Day 8's scale.


$$
\gamma=-0.0356\pm0.0202\ \mathrm{mag}
$$

This is consistent with the published value of $-0.06$ mag. A step of that size would be about $3\sigma$, so my sample could potentially detect it. However, my measured step is only about $1.8\sigma$ from zero, so there is not strong evidence for a step in my data.

### 2e - Testing your pipeline (8 pts)

Design one test of your pipeline that would come out differently if your pipeline were wrong, and run it. State **before** you run it what result would count as a failure. One suggestion: generate fake supernovae at your redshifts, stretches and colors from parameters you choose (including a $\gamma$ you choose), fit 30 or more such datasets (maximum-likelihood fits with curvature error bars are fine here; you don't need MCMC for each), and check how often your 68% intervals contain the truth.

In [ ]:
# 2e: your work here

true=map_p.copy();true[3]=-.06
old_mb=mb.copy()
rng=np.random.default_rng(92)
recovered=[]
errors=[]
for j in range(60):
    var=smb**2+true[1]**2*sx**2+true[2]**2*sc**2+2*true[1]*cmx-2*true[2]*cmc-2*true[1]*true[2]*cxc+true[4]**2+(5/np.log(10)*250/(299792.458*z))**2
    if np.any(var<=0):raise ValueError('Negative variance')
    mb=(mu+true[0]-true[1]*x1+true[2]*color+true[3]*I)+rng.normal(0,np.sqrt(var))
    fakefit=minimize(lambda p:neglog(p,keep),true,method='Nelder-Mead',options={'maxiter':4000})
    if not fakefit.success: print('Optimizer warning on simulation',j)
    H=hessian(lambda p:neglog(p,keep),fakefit.x,np.array([.002,.002,.01,.002,.002]))
    try: se=np.sqrt(np.linalg.inv(H)[3,3])
    except np.linalg.LinAlgError: se=np.nan
    recovered.append(fakefit.x[3]);errors.append(se)
mb=old_mb
recovered=np.array(recovered);errors=np.array(errors)
good=np.isfinite(errors)
covered=np.mean(abs(recovered[good]-true[3])<=errors[good])
print('Successful curvature estimates:',sum(good),'of',len(good))
print('68% interval coverage:',covered)
print('PREDECLARED PASS?', .55<=covered<=.80)
plt.hist(recovered[good],bins=16);plt.axvline(true[3],ls='--');plt.xlabel('Recovered delta (mag)');plt.show()

**ANSWER:**

I tested my pipeline by generating 60 simulated supernova datasets with a true host-mass step of $\gamma=-0.06$ mag. I decided that coverage outside 55% to 80% would count as a failure.

All 60 fits were successful, and 75% of the 68% confidence intervals contained the true value. This passed my test and suggests that the pipeline gives reasonable uncertainty estimates for the simulated data.

## Part 3: The referee report (25 pts)

White dwarfs are the cores of dead stars like the Sun. Your file `p3_wd_masses.csv` holds 600 real DA (hydrogen-atmosphere) white dwarfs from the 100 pc sample of [Kilic et al. (2020)](https://arxiv.org/abs/2006.00323), ApJ 898, 84: `name`, `teff_K`, `mass_Msun`, `mass_err_Msun`. Most single white dwarfs have masses near $0.6\,M_\odot$; Kilic et al. report a narrow peak there plus an excess of more massive ones, around $0.7$-$0.9\,M_\odot$, which merger products alone do not explain.

The notebook `labs/midterm/ai_solution.ipynb` fits your masses with a two-population model using `emcee`. Run it from that directory, after setting `NETID` in its first cell. It runs top to bottom and the plots look nice, but it's wrong in at least three substantive ways - and if you don't look at your data, you'll fool yourself the same way.

### 3a - The report (20 pts)

1. **Identify** at least three substantive errors (not style; "it used 32 walkers and I'd use 48" is not one).
2. **Demonstrate** each on *your* file: show, with a number or a plot, what the error does to the answer. A demonstration can come out negative: if an error doesn't move your numbers, show that, and say what it would take for it to matter.
3. **State** the correct treatment, and run it where you can.

In [ ]:
# 3a: your work here
import numpy as np
from scipy.optimize import minimize
from scipy.stats import norm

wd = load('p3_wd_masses')
mass = wd['mass_Msun'].to_numpy()
err = wd['mass_err_Msun'].to_numpy()

def fit(m, e, start):
    def loss(p):
        m1, m2, s1, s2, f = p
        if not (.2 < m1 < m2 < 1.4 and .005 < s1 < .5 and .005 < s2 < .5 and 0 < f < 1):
            return 1e20
        a = np.log(f) + norm.logpdf(m, m1, np.sqrt(s1**2 + e**2))
        b = np.log(1-f) + norm.logpdf(m, m2, np.sqrt(s2**2 + e**2))
        return -np.sum(np.logaddexp(a, b))
    return minimize(loss, start, method='Nelder-Mead', options={'maxiter':12000})


print('Error 1: Removing high-mass stars')
print('Error 2: Ignoring measurement errors')
print('Error 3: Using population widths as uncertainties on the means')


full = fit(mass, err, [.6, .72, .033, .21, .42])

med = np.median(mass)
sd = np.std(mass)
keep = np.abs(mass - med) < 3*sd
cut = fit(mass[keep], err[keep], full.x)

print('\nError 1:')
print('Stars removed:', np.sum(~keep))
print('Mean 2 before:', full.x[1])
print('Mean 2 after:', cut.x[1])

ignore = fit(mass, np.zeros(len(mass)), full.x)

print('\nError 2:')
print('Width 1 without errors:', ignore.x[2])
print('Width 1 with errors:', full.x[2])

rng = np.random.default_rng(24)
means = []

for i in range(40):
    ids = rng.choice(len(mass), len(mass), replace=True)
    result = fit(mass[ids], err[ids], full.x)
    if result.success:
        means.append(result.x[:2])

means = np.array(means)
unc = np.std(means, axis=0, ddof=1)

print('\nError 3:')
print('Population widths:', full.x[2:4])
print('Uncertainties on means:', unc)

print('\nCorrected results:')
print('All stars used:', len(mass))
print('Mean 1:', full.x[0], '+/-', unc[0], 'Msun')
print('Mean 2:', full.x[1], '+/-', unc[1], 'Msun')
print('Width 1:', full.x[2], 'Msun')
print('Width 2:', full.x[3], 'Msun')
print('Fraction in population 1:', full.x[4])



**REFEREE REPORT:**

**1. Removing high-mass stars:** The AI removed 10 stars using a $3\sigma$ cut. This changed the second population mean from $0.7180$ to $0.6988\ M_\odot$. This is wrong because those stars might belong to the high-mass population. I corrected this by fitting all 600 stars.

**2. Ignoring measurement errors:** The AI ignored the mass measurement errors. Without them, the first population width was $0.0363\ M_\odot$, compared with $0.0331\ M_\odot$ when including them. I corrected this by adding each star's measurement-error variance to the population variance.

**3. Incorrect uncertainties:** The AI used population widths as uncertainties on the means. The widths were $0.0331$ and $0.2121\ M_\odot$, but my bootstrap gave uncertainties of only $0.0031$ and $0.0147\ M_\odot$. I corrected this by using bootstrap estimates for the uncertainties on the means.

### 3b - AI-use appendix (5 pts)

1. Which tools did you use, for what, what did they get wrong, and how did you catch it? "I used ChatGPT to help" is useless; "none" is fine if that's true. I promise I don't care which - just be honest about what you did.
2. What would have gone wrong in your answer if you had not checked the AI's output? Name the specific thing, in two or three sentences.

**AI-USE APPENDIX:**

1. I used ChatGPT to help write and organize my code, understand parts of the assignment, and format my written answers in LaTeX for Jupyter Notebook. Some of the code had errors, such as missing imports, an undefined Hessian function, and an ArviZ function that did not work. I caught these errors by running the code, reading the error messages, and asking ChatGPT to help fix them. I also checked the results using the plots, and resulting calculations.

2. If I had not checked the AI's output, I could have submitted code that did not run properly and reported incorrect results. In Part 3, I could have removed important high-mass white dwarfs and confused population widths with uncertainties on the means, which would have affected my conclusions. ChatGPT also made a mistake in Part 2c by fixing $\beta$ and refitting $\alpha$ instead of fixing $M$ and refitting $\gamma$, which was what the question actually asked. I caught this by comparing the code with the assignment instructions and corrected it before submitting.

Get started. You know how to do all of this. If not, there's something in a lab or class notebook that can help. Slack me with questions. Best of luck.